In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Single Head Attention

In [2]:
def attention(Q, K, V):
  d_k = Q.shape[-1]
  scale = 1 / math.sqrt(d_k)
  S = Q @ K.mT * scale
  P = torch.softmax(S, dim = -1)
  O = P @ V
  return O

In [3]:
N = 32
d_model = 512
d_k = d_q = d_v = d_model

In [4]:
K = torch.rand(N, d_k)
Q = torch.rand(N, d_q)
V = torch.rand(N, d_v)

In [5]:
O = attention(Q, K, V)
ref = F.scaled_dot_product_attention(Q, K, V)

In [6]:
torch.allclose(O, ref, atol=1e-5)

True

# Multi-Head Attention

In [7]:
class MultiHeadedAttention(nn.Module):
  def __init__(self, num_heads, d_model, bias=False):
    super().__init__()
    assert d_model % num_heads == 0, "model dimension must be divisible by the number of heads"

    self.num_heads = num_heads
    self.d_model = d_model
    self.d_k = d_model // num_heads

    # weight matrix projections
    self.W_Q = nn.Linear(d_model, d_model, bias=bias)
    self.W_K = nn.Linear(d_model, d_model, bias=bias)
    self.W_V = nn.Linear(d_model, d_model, bias=bias)
    self.W_O = nn.Linear(d_model, d_model, bias=bias)

  # attention
  def scaled_dot_product_attention(self, Q, K, V, mask=None):
    scale = 1 / math.sqrt(self.d_k)
    S = Q @ K.transpose(-2, -1) * scale

    if mask is not None:
      S = S.masked_fill(mask, float('-inf'))

    P = torch.softmax(S, dim = -1)
    O = P @ V
    return O

  # forward pass
  def forward(self, x, mask=None):
    # apply projections
    Q = self.W_Q(x)
    K = self.W_K(x)
    V = self.W_V(x)

    # split feature dim into heads
    Q = Q.view(*x.shape[:-1], self.num_heads, self.d_k).transpose(-3, -2)
    K = K.view(*x.shape[:-1], self.num_heads, self.d_k).transpose(-3, -2)
    V = V.view(*x.shape[:-1], self.num_heads, self.d_k).transpose(-3, -2)

    O = self.scaled_dot_product_attention(Q, K, V, mask)
    # undo transpose and reshape, merges d_k and h back to d_model, reshape left it non-contiguous
    O = O.transpose(-3, -2).reshape(*x.shape[:-1], self.d_model)
    O = self.W_O(O)

    return O

In [8]:
h = 8
mha = MultiHeadedAttention(h, d_model)

In [9]:
B = 4
x = torch.rand(N, d_model)
x_batched = torch.rand(B, N, d_model)

print(mha(x).shape, ) 
print(mha(x_batched).shape)

torch.Size([32, 512])
torch.Size([4, 32, 512])


In [10]:
print(torch.allclose(mha(x_batched[0]), mha(x_batched)[0], atol=1e-5))

True


In [11]:
mask = torch.triu(torch.ones(N, N, dtype=torch.bool), diagonal=1)
mask

tensor([[False,  True,  True,  ...,  True,  True,  True],
        [False, False,  True,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        ...,
        [False, False, False,  ..., False,  True,  True],
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False, False]])

In [12]:
x2 = x.clone(); x2[N//2:] = torch.rand(N - N//2, d_model)
x2

tensor([[4.8645e-01, 9.2603e-01, 2.2589e-01,  ..., 1.5247e-01, 9.5405e-01,
         9.5313e-01],
        [2.7472e-01, 6.8803e-01, 6.7423e-01,  ..., 5.9872e-01, 4.5571e-01,
         9.8883e-01],
        [7.8711e-01, 4.0103e-01, 7.7922e-01,  ..., 1.3170e-01, 5.6400e-02,
         8.2049e-01],
        ...,
        [7.6693e-01, 3.9455e-02, 9.0567e-01,  ..., 5.4514e-02, 9.8628e-01,
         1.8147e-03],
        [5.4391e-01, 1.8607e-01, 4.7962e-01,  ..., 5.6587e-01, 1.2474e-01,
         5.1295e-01],
        [6.3388e-01, 8.9050e-01, 7.2199e-01,  ..., 8.2793e-01, 8.1231e-01,
         9.6691e-04]])

In [13]:
torch.allclose(mha(x, mask)[:N//2], mha(x2, mask)[:N//2], atol=1e-5)

True